# SIGN NAV PLUS R3：まず1走行を読む

CPUで実行できます。GPUとAPIキーは不要です。アップロードするのは「PAI_SIGN_NAV_PLUS_R3_コード一式.zip」です。

制作環境で同じPython処理を確認しています。Colab上の実行確認は行っていません。

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded=files.upload()
zips=[name for name in uploaded if name.lower().endswith(".zip")]
if len(zips)!=1: raise ValueError("コードZIPを1つだけ選んでください。")
base=Path("/content").resolve()
with zipfile.ZipFile(zips[0]) as z:
    if "pai_signnav_plus/main.py" not in z.namelist(): raise ValueError("提出セットではなくコード一式ZIPを選んでください。")
    for name in z.namelist():
        if not (base/name).resolve().is_relative_to(base): raise ValueError("不正なパスです。")
    z.extractall(base)
print("準備できました。")

In [ ]:
import os, subprocess, sys
os.chdir("/content/pai_signnav_plus")
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements-core.txt"],check=True)

## 1. 方式Dを1回実行

既存の比較記録を上書きしない専用フォルダーへ保存します。

In [ ]:
subprocess.run([sys.executable,"explain_run.py","--case","D","--out","results/my_first_run"],check=True)

In [ ]:
import json
from IPython.display import display, Markdown, Image
p=Path("results/my_first_run")
m=json.loads((p/"mission.json").read_text())
names=["左","直進","右"]
for o in m["observations"]:
    j=o["junction"]
    view=o["view"]+1
    correct=names[m["result"]["true_labels"][j]]
    predicted=names[o["prediction"]["label"]]
    action=o["action"]
    display(Markdown(f"### 標識{j+1}・位置{view}\n正解：{correct} ／ AI：{predicted} ／ 判断：{action}"))
    display(Image(filename=str(p/o["image"]),width=200))


## 2. 記録の点検（任意）

VERIFY=Trueにして実行します。480走行を最初から再走行する処理ではなく、記録と判定の点検と代表4例の再走行です。

In [ ]:
VERIFY=False
if VERIFY:
    subprocess.run([sys.executable,"verify.py"],check=True)

## 3. 全480走行を最初から行う場合だけ

RUN_ALL=Trueにします。保存済みbenchmarkとデモ記録を再作成するので、元のコードZIPを残してください。

In [ ]:
RUN_ALL=False
if RUN_ALL:
    subprocess.run([sys.executable,"experiments.py"],check=True)

終了後はColabのランタイムを接続解除します。必要な実行結果を先に保存してください。